In [ ]:
import openmeteo_requests
import requests 
import pandas as pd
import requests_cache
from retry_requests import retry

In [98]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

In [99]:
lat_list = [info["lat"] for info in london_regions.values()]
long_list =  [info["long"] for info in london_regions.values()]

In [100]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": london_regions["East London"]["lat"],
	"longitude": london_regions["East London"]["long"],
	"daily": ["sunrise", "sunset"],
	"hourly": ["temperature_2m", "apparent_temperature", "precipitation_probability", "rain", "weather_code", "wind_speed_10m", "wind_gusts_10m"],
    "forecast_days": 14,
	"timezone": "auto",
}
responses = openmeteo.weather_api(url, params=params)

In [101]:
len(responses)

1

In [102]:
#Take Central London for example 
response = responses[0]


In [103]:
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

Coordinates: 51.540000915527344°N -2.384185791015625e-07°E
Elevation: 10.0 m asl
Timezone: b'Europe/London'None
Timezone difference to GMT+0: 0s


In [104]:
# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_apparent_temperature = hourly.Variables(1).ValuesAsNumpy()
hourly_precipitation_probability = hourly.Variables(2).ValuesAsNumpy()
hourly_rain = hourly.Variables(3).ValuesAsNumpy()
hourly_weather_code = hourly.Variables(4).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(5).ValuesAsNumpy()
hourly_wind_gusts_10m = hourly.Variables(6).ValuesAsNumpy()

In [105]:
hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}


In [106]:
hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["apparent_temperature"] = hourly_apparent_temperature
hourly_data["precipitation_probability"] = hourly_precipitation_probability
hourly_data["rain"] = hourly_rain
hourly_data["weather_code"] = hourly_weather_code
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
hourly_data["wind_gusts_10m"] = hourly_wind_gusts_10m

In [107]:
hourly_dataframe = pd.DataFrame(data = hourly_data)
hourly_dataframe

,date,temperature_2m,apparent_temperature,precipitation_probability,rain,weather_code,wind_speed_10m,wind_gusts_10m
0,2026-02-18 00:00:00+00:00,2.6825,-1.026186,0.0,0.0,3.0,9.360000,18.719999
1,2026-02-18 01:00:00+00:00,2.6325,-1.087992,0.0,0.0,3.0,9.746631,21.240000
2,2026-02-18 02:00:00+00:00,2.6325,-0.907292,0.0,0.0,3.0,9.387650,21.959999
3,2026-02-18 03:00:00+00:00,3.2825,-0.528567,0.0,0.0,3.0,11.367109,24.840000
4,2026-02-18 04:00:00+00:00,3.4325,-0.731637,0.0,0.0,3.0,13.009903,29.519999
...,...,...,...,...,...,...,...,...
331,2026-03-03 19:00:00+00:00,11.0495,9.399888,19.0,0.0,3.0,8.209263,21.959999
332,2026-03-03 20:00:00+00:00,10.9495,9.334995,18.0,0.0,3.0,7.172949,19.799999
333,2026-03-03 21:00:00+00:00,10.8495,9.408513,18.0,0.0,3.0,5.815978,17.639999
334,2026-03-03 22:00:00+00:00,10.7495,9.702425,17.0,0.0,3.0,3.563818,14.759999


In [108]:
# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_sunrise = daily.Variables(0).ValuesInt64AsNumpy()
daily_sunset = daily.Variables(1).ValuesInt64AsNumpy()

In [109]:
daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	end =  pd.to_datetime(daily.TimeEnd() + response.UtcOffsetSeconds(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}

In [110]:
daily_data["sunrise"] = daily_sunrise
daily_data["sunset"] = daily_sunset

daily_dataframe = pd.DataFrame(data = daily_data)

In [112]:
daily_dataframe

,date,sunrise,sunset
0,2026-02-18 00:00:00+00:00,1771398467,1771435194
1,2026-02-19 00:00:00+00:00,1771484745,1771521704
2,2026-02-20 00:00:00+00:00,1771571022,1771608213
3,2026-02-21 00:00:00+00:00,1771657299,1771694722
4,2026-02-22 00:00:00+00:00,1771743574,1771781231
5,2026-02-23 00:00:00+00:00,1771829849,1771867740
6,2026-02-24 00:00:00+00:00,1771916123,1771954249
7,2026-02-25 00:00:00+00:00,1772002396,1772040757
8,2026-02-26 00:00:00+00:00,1772088669,1772127264
9,2026-02-27 00:00:00+00:00,1772174942,1772213771


In [115]:
import pandas as pd

# df has columns: date, sunrise, sunset
daily_dataframe["date"] = pd.to_datetime(daily_dataframe["date"], utc=True)

daily_dataframe["sunrise_dt"] = pd.to_datetime(daily_dataframe["sunrise"], unit="s", utc=True).dt.tz_convert("Europe/London")
daily_dataframe["sunset_dt"]  = pd.to_datetime(daily_dataframe["sunset"],  unit="s", utc=True).dt.tz_convert("Europe/London")

# optional: keep just the clock time
daily_dataframe["sunrise_time"] = daily_dataframe["sunrise_dt"].dt.strftime("%H:%M")
daily_dataframe["sunset_time"]  = daily_dataframe["sunset_dt"].dt.strftime("%H:%M")

daily_dataframe = daily_dataframe[["date", "sunrise_time", "sunset_time"]]

In [116]:
daily_dataframe

,date,sunrise_time,sunset_time
0,2026-02-18 00:00:00+00:00,07:07,17:19
1,2026-02-19 00:00:00+00:00,07:05,17:21
2,2026-02-20 00:00:00+00:00,07:03,17:23
3,2026-02-21 00:00:00+00:00,07:01,17:25
4,2026-02-22 00:00:00+00:00,06:59,17:27
5,2026-02-23 00:00:00+00:00,06:57,17:29
6,2026-02-24 00:00:00+00:00,06:55,17:30
7,2026-02-25 00:00:00+00:00,06:53,17:32
8,2026-02-26 00:00:00+00:00,06:51,17:34
9,2026-02-27 00:00:00+00:00,06:49,17:36


In [117]:
hourly_dataframe.head(25)

,date,temperature_2m,apparent_temperature,precipitation_probability,rain,weather_code,wind_speed_10m,wind_gusts_10m
0,2026-02-18 00:00:00+00:00,2.6825,-1.026186,0.0,0.0,3.0,9.360000,18.719999
1,2026-02-18 01:00:00+00:00,2.6325,-1.087992,0.0,0.0,3.0,9.746631,21.240000
2,2026-02-18 02:00:00+00:00,2.6325,-0.907292,0.0,0.0,3.0,9.387650,21.959999
3,2026-02-18 03:00:00+00:00,3.2825,-0.528567,0.0,0.0,3.0,11.367109,24.840000
4,2026-02-18 04:00:00+00:00,3.4325,-0.731637,0.0,0.0,3.0,13.009903,29.519999
5,2026-02-18 05:00:00+00:00,3.5325,-0.954891,0.0,0.0,3.0,14.759999,31.319998
6,2026-02-18 06:00:00+00:00,3.6825,-0.631172,0.0,0.0,3.0,13.708391,32.760002
7,2026-02-18 07:00:00+00:00,3.5325,-1.027301,0.0,0.0,3.0,15.256526,34.560001
8,2026-02-18 08:00:00+00:00,3.6825,-0.696785,0.0,0.0,3.0,13.779114,32.760002
9,2026-02-18 09:00:00+00:00,4.2825,-0.309722,0.0,0.0,3.0,15.596767,33.119999


In [131]:
from datetime import time

def is_playable_weather(row) -> bool:
    return (
        row["rain"] <= 0.5 and
        row["precipitation_probability"] < 60 and
        row["wind_speed_10m"] < 25 and
        row["wind_gusts_10m"] < 45 and
        row["apparent_temperature"] > 0 and
        row["weather_code"] < 61
    )

def playable_conditions(row) -> bool:
    t = row["date"].time()
    dow = row["date"].day_of_week  # Mon=0 ... Sun=6

    if dow in [0, 1, 2, 3, 4]:  # weekdays
        if t == time(19, 0):
            return is_playable_weather(row)

    if dow in [5, 6]:  # weekend
        if time(10, 0) <= t <= time(15, 0):
            return is_playable_weather(row)

    return False

In [132]:
hourly_dataframe

,date,temperature_2m,apparent_temperature,precipitation_probability,rain,weather_code,wind_speed_10m,wind_gusts_10m
0,2026-02-18 00:00:00+00:00,2.6825,-1.026186,0.0,0.0,3.0,9.360000,18.719999
1,2026-02-18 01:00:00+00:00,2.6325,-1.087992,0.0,0.0,3.0,9.746631,21.240000
2,2026-02-18 02:00:00+00:00,2.6325,-0.907292,0.0,0.0,3.0,9.387650,21.959999
3,2026-02-18 03:00:00+00:00,3.2825,-0.528567,0.0,0.0,3.0,11.367109,24.840000
4,2026-02-18 04:00:00+00:00,3.4325,-0.731637,0.0,0.0,3.0,13.009903,29.519999
...,...,...,...,...,...,...,...,...
331,2026-03-03 19:00:00+00:00,11.0495,9.399888,19.0,0.0,3.0,8.209263,21.959999
332,2026-03-03 20:00:00+00:00,10.9495,9.334995,18.0,0.0,3.0,7.172949,19.799999
333,2026-03-03 21:00:00+00:00,10.8495,9.408513,18.0,0.0,3.0,5.815978,17.639999
334,2026-03-03 22:00:00+00:00,10.7495,9.702425,17.0,0.0,3.0,3.563818,14.759999


In [142]:
playable_filter = hourly_dataframe.apply(playable_conditions, axis = 1)

In [143]:
playable_times = hourly_dataframe[playable_filter]

In [149]:
playable_times

,date,temperature_2m,apparent_temperature,precipitation_probability,rain,weather_code,wind_speed_10m,wind_gusts_10m
43,2026-02-19 19:00:00+00:00,6.532500,3.349577,3.0,0.0,3.0,10.948973,23.039999
67,2026-02-20 19:00:00+00:00,9.982500,6.795741,3.0,0.0,1.0,14.917212,37.439999
82,2026-02-21 10:00:00+00:00,11.732500,9.173289,1.0,0.0,3.0,15.978484,37.079998
83,2026-02-21 11:00:00+00:00,13.332500,11.011416,3.0,0.0,3.0,14.843180,43.560001
106,2026-02-22 10:00:00+00:00,12.182500,10.360155,25.0,0.1,3.0,13.783817,36.360001
107,2026-02-22 11:00:00+00:00,12.682500,10.963277,24.0,0.1,3.0,13.392774,38.160000
108,2026-02-22 12:00:00+00:00,13.032500,11.383965,23.0,0.1,3.0,12.904882,38.519997
109,2026-02-22 13:00:00+00:00,13.232500,11.686224,24.0,0.1,3.0,12.245293,36.000000
110,2026-02-22 14:00:00+00:00,13.282500,11.806124,25.0,0.1,3.0,11.503113,31.680000
111,2026-02-22 15:00:00+00:00,13.132501,11.541042,27.0,0.1,3.0,11.275530,28.799999


In [150]:
import pandas as pd

# ---------- 1) Playability + scoring ----------

def _clamp(x, lo, hi):
    return max(lo, min(hi, x))

def playability_label(row) -> str:
    # "maybe" band
    if row["rain"] > 0.1:
        return "maybe"
    if row["precipitation_probability"] >= 30:
        return "maybe"
    if row["wind_speed_10m"] >= 18:
        return "maybe"
    if row["wind_gusts_10m"] >= 35:
        return "maybe"
    if row["apparent_temperature"] < 3:
        return "maybe"
    if row["weather_code"] >= 51:  # drizzle family
        return "maybe"

    return "playable"

def session_score(row) -> int:
    """
    0–100 "tennis vibes" score.
    Warmer, drier, less wind/gusts => higher.
    """
    # temp: 0°C meh, 10°C great, 15°C max
    temp = _clamp((row["apparent_temperature"] - 0) / 15, 0, 1)

    # precip prob: 0% best, 60% bad
    pop = 1 - _clamp(row["precipitation_probability"] / 60, 0, 1)

    # wind: 0 best, 25 bad
    wind = 1 - _clamp(row["wind_speed_10m"] / 25, 0, 1)

    # gusts: 0 best, 45 bad
    gust = 1 - _clamp(row["wind_gusts_10m"] / 45, 0, 1)

    # penalties
    rain_penalty = _clamp(row["rain"] / 1.0, 0, 1) if row["rain"] > 0 else 0

    wc = int(row["weather_code"])
    code_penalty = 0
    if 51 <= wc <= 57:
        code_penalty = 0.15
    elif wc >= 61:
        code_penalty = 0.60

    raw = (
        0.35 * temp +
        0.25 * pop +
        0.20 * wind +
        0.20 * gust
    )

    raw = raw - 0.45 * rain_penalty - code_penalty
    return int(round(100 * _clamp(raw, 0, 1)))


# ---------- 2) Emojis + formatting ----------

def sky_emoji(weather_code: int) -> str:
    wc = int(weather_code)
    if wc in (0, 1):
        return "☀️"
    if wc == 2:
        return "🌤️"
    if wc == 3:
        return "☁️"
    if wc in (45, 48):
        return "🌫️"
    if 51 <= wc <= 57:
        return "🌦️"   # drizzle
    if 61 <= wc <= 67:
        return "☔"    # rain
    if 71 <= wc <= 77:
        return "❄️"    # snow
    if 95 <= wc <= 99:
        return "⛈️"    # thunderstorm
    return "🌥️"

def status_emoji(label: str, score: int, is_best: bool) -> str:
    # "best day" gets the 🔥 crown treatment
    if is_best:
        return "🔥"
    if label == "playable" and score >= 75:
        return "✅"
    if label == "playable":
        return "✅"
    if label == "maybe":
        return "⚠️"
    return "❌"

def add_wind_emoji(wind_speed: float, gusts: float) -> str:
    return "🌬️" if (wind_speed >= 18 or gusts >= 40) else ""

def add_rain_emoji(rain: float, pop: float, weather_code: int) -> str:
    if rain > 0.1:
        return "☔"
    if pop >= 30 or (51 <= int(weather_code) <= 57):
        return "🌦️"
    return ""


# ---------- 3) Merge consecutive hours into "windows" ----------

def build_windows(df: pd.DataFrame, tz: str = "Europe/London") -> pd.DataFrame:
    """
    Converts hourly rows into windows per calendar date by merging consecutive hours.
    Window score = max session score within the window (and uses that row as the representative row).
    """
    dfx = df.copy()

    # Ensure datetime and timezone
    dfx["date"] = pd.to_datetime(dfx["date"], utc=True, errors="coerce")
    if tz:
        dfx["date"] = dfx["date"].dt.tz_convert(tz)

    dfx = dfx.sort_values("date").reset_index(drop=True)

    # compute label+score per row
    dfx["label"] = dfx.apply(playability_label, axis=1)
    dfx["score"] = dfx.apply(session_score, axis=1)

    # (Optional) Drop any "no" rows if they slipped in
    dfx = dfx[dfx["label"] != "no"].copy()
    if dfx.empty:
        return dfx

    windows = []
    current = [dfx.iloc[0]]

    def flush(block_rows):
        block = pd.DataFrame(block_rows)
        # representative row = highest score
        rep = block.sort_values("score", ascending=False).iloc[0].to_dict()

        # label for block: if any maybe -> maybe else playable
        label_rank = {"playable": 0, "maybe": 1}
        worst = max(block["label"], key=lambda x: label_rank.get(x, 999))
        rep["block_label"] = worst

        start = block_rows[0]["date"]
        last = block_rows[-1]["date"]
        end = last + pd.Timedelta(hours=1)

        rep["start"] = start
        rep["end"] = end
        rep["day"] = start.date()

        windows.append(rep)

    for i in range(1, len(dfx)):
        prev = dfx.iloc[i - 1]["date"]
        cur = dfx.iloc[i]["date"]

        same_day = prev.date() == cur.date()
        consecutive = (cur - prev) == pd.Timedelta(hours=1)

        if same_day and consecutive:
            current.append(dfx.iloc[i])
        else:
            flush(current)
            current = [dfx.iloc[i]]

    flush(current)

    return pd.DataFrame(windows).sort_values("score", ascending=False).reset_index(drop=True)


# ---------- 4) Build the Telegram-style message ----------

def tennis_telegram_summary(df: pd.DataFrame, tz: str = "Europe/London") -> str:
    windows = build_windows(df, tz=tz)
    if windows.empty:
        return "🎾 No playable windows found in your filtered data."

    best = windows.iloc[0]

    def fmt_day(dt):
        return dt.strftime("%a %d %b")

    def fmt_time_range(start, end):
        return f"{start.strftime('%H:%M')}–{end.strftime('%H:%M')}"

    # Header "Best session"
    best_sky = sky_emoji(best["weather_code"])
    best_wind = add_wind_emoji(best["wind_speed_10m"], best["wind_gusts_10m"])
    header = (
        "🎾 *Tennis Playability Report* (from your filtered playable rows)\n\n"
        f"🏆 *Best session (overall):* *{fmt_day(best['start'])}, {fmt_time_range(best['start'], best['end'])}* "
        f"{best_sky}{best_wind}\n"
        f"Feels like *{best['apparent_temperature']:.0f}°C* • PoP *{best['precipitation_probability']:.0f}%* "
        f"• Rain *{best['rain']:.1f}mm* • Wind/Gust *{best['wind_speed_10m']:.0f}/{best['wind_gusts_10m']:.0f}*\n\n"
    )

    # Body list
    body_lines = ["*Upcoming playable windows (ranked best → worst)*"]
    for idx, row in windows.iterrows():
        is_best = idx == 0
        label = row["block_label"]
        score = int(row["score"])

        status = status_emoji(label, score, is_best)
        sky = sky_emoji(row["weather_code"])
        wind = add_wind_emoji(row["wind_speed_10m"], row["wind_gusts_10m"])
        rain_em = add_rain_emoji(row["rain"], row["precipitation_probability"], row["weather_code"])

        line = (
            f"{status} *{fmt_day(row['start'])}* {sky}{rain_em}{wind}  | "
            f"*{fmt_time_range(row['start'], row['end'])}*  | "
            f"feels *{row['apparent_temperature']:.0f}°C* | "
            f"PoP *{row['precipitation_probability']:.0f}%* | "
            f"rain *{row['rain']:.1f}mm* | "
            f"gust *{row['wind_gusts_10m']:.0f}*  "
            f"(_score {score}/100_)"
        )
        body_lines.append(line)

    # Legend
    legend = (
        "\n\n_Emoji legend:_ 🔥 best vibes • ✅ solid • ⚠️ playable but annoying • "
        "☀️/🌤️/☁️ sky • 🌦️ drizzle risk • ☔ measurable rain • 🌬️ gusty"
    )

    # Telegram supports MarkdownV2 or HTML; this string is Markdown-ish.
    return header + "\n".join(body_lines) + legend


# ---------- 5) Example usage ----------
# summary_text = tennis_telegram_summary(df_playable)   # df_playable = your filtered dataframe
# print(summary_text)

In [151]:
summary_text = tennis_telegram_summary(playable_times)

In [152]:
print(summary_text)

🎾 *Tennis Playability Report* (from your filtered playable rows)

🏆 *Best session (overall):* *Tue 03 Mar, 19:00–20:00* ☁️
Feels like *9°C* • PoP *19%* • Rain *0.0mm* • Wind/Gust *8/22*

*Upcoming playable windows (ranked best → worst)*
🔥 *Tue 03 Mar* ☁️  | *19:00–20:00*  | feels *9°C* | PoP *19%* | rain *0.0mm* | gust *22*  (_score 63/100_)
⚠️ *Sat 21 Feb* ☁️🌬️  | *10:00–12:00*  | feels *11°C* | PoP *3%* | rain *0.0mm* | gust *44*  (_score 58/100_)
✅ *Mon 23 Feb* ☁️  | *19:00–20:00*  | feels *8°C* | PoP *12%* | rain *0.0mm* | gust *31*  (_score 55/100_)
⚠️ *Sun 22 Feb* ☁️☔  | *10:00–16:00*  | feels *12°C* | PoP *25%* | rain *0.1mm* | gust *32*  (_score 54/100_)
✅ *Thu 19 Feb* ☁️  | *19:00–20:00*  | feels *3°C* | PoP *3%* | rain *0.0mm* | gust *23*  (_score 53/100_)
⚠️ *Fri 20 Feb* ☀️  | *19:00–20:00*  | feels *7°C* | PoP *3%* | rain *0.0mm* | gust *37*  (_score 51/100_)
⚠️ *Fri 27 Feb* 🌦️🌦️  | *19:00–20:00*  | feels *8°C* | PoP *32%* | rain *0.0mm* | gust *19*  (_score 40/100_)
⚠️ *Sa